# Partie 6 — Prime finale et soumission Kaggle

La prime pure d'un client est le produit des deux modèles :

$$\widehat{\text{Prime}}(x) = \hat p(x) \times \hat g(x)$$

où $\hat p$ est la probabilité de sinistre (notebook 01) et $\hat g$ le montant attendu si sinistre (notebook 02).

**Métrique Kaggle** (compétition CY Tech 2027, vérifiée par mon binôme sur la page de la compétition) : la **RMSE de la prime** sur le test (la consigne parle de MSE ; les deux classent les modèles dans le même ordre). Le fichier de soumission a deux colonnes : `index` et `pred`.

**Évaluation locale** : on calcule la même RMSE sur le train, avec les prédictions **hors pli** (5 plis par client, colonne `pli`). Le pli 1 correspond exactement au jeu de validation du notebook 01.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Racine du projet : le dossier qui contient Data/ (lancement depuis notebooks/ ou depuis la racine)
RACINE = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "Data" / "train.csv").exists())
PREDICTIONS = RACINE / "Data" / "predictions"

train = pd.read_csv(RACINE / "Data" / "train.csv", dtype={"code_postal": str})
test = pd.read_csv(RACINE / "Data" / "test.csv", dtype={"code_postal": str})
oof = pd.read_csv(PREDICTIONS / "binome_composantes_oof_train.csv")   # prédictions hors pli sur le train
binome_test = pd.read_csv(PREDICTIONS / "binome_composantes_test.csv")  # prédictions sur le test Kaggle
assert (oof["index"].values == train["index"].values).all() and (binome_test["index"].values == test["index"].values).all()
print("train :", train.shape, "| test :", test.shape, "| plis :", sorted(oof["pli"].unique()))

train : (50000, 33) | test : (50000, 28) | plis : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


## 1. Toutes les combinaisons fréquence × gravité disponibles

Trois fréquences : la régression logistique de référence (ma version d'avant l'audit, reprise par le binôme), ma logistique actuelle (notebook 01) et la logistique de référence corrigée par des courbes (binôme). Trois gravités. On compare la RMSE locale hors pli avec le score Kaggle public quand il existe.

In [2]:
# Ma fréquence (notebook 01) : mêmes plis, mêmes contrats que les prédictions du binôme
ma_freq_oof = pd.read_csv(PREDICTIONS / "frequence_logistique_oof_train.csv")
ma_freq_test = pd.read_csv(PREDICTIONS / "frequence_logistique_test.csv")
assert (ma_freq_oof["index"].values == oof["index"].values).all() and (ma_freq_oof["pli"].values == oof["pli"].values).all()
assert (ma_freq_test["id"].values == binome_test["index"].values).all()
oof["proba_ma_logistique"] = ma_freq_oof["proba_pred"].values
binome_test["proba_ma_logistique"] = ma_freq_test["proba_pred"].values

# Les candidats du notebook 01 (sections 4.12 à 4.15)
for cle in ["selection6", "metier", "metier_novice2", "14variables", "final"]:
    cand_oof = pd.read_csv(PREDICTIONS / f"frequence_{cle}_oof_train.csv")
    cand_test = pd.read_csv(PREDICTIONS / f"frequence_{cle}_test.csv")
    assert (cand_oof["pli"].values == oof["pli"].values).all() and (cand_test["id"].values == binome_test["index"].values).all()
    oof[f"proba_{cle}"] = cand_oof["proba_pred"].values
    binome_test[f"proba_{cle}"] = cand_test["proba_pred"].values

FREQUENCES = {"logistique de référence": "proba_logistique_reference",
              "ma logistique (notebook 01)": "proba_ma_logistique",
              "sélection 6 variables (notebook 01)": "proba_selection6",
              "actuel + métier, m = 1000 (notebook 01)": "proba_metier",
              "23 variables (notebook 01, 4.13)": "proba_metier_novice2",
              "14 variables (notebook 01, 4.14)": "proba_14variables",
              "modèle final E, 17 variables (notebook 01, 4.15)": "proba_final",
              "logistique + courbes": "proba_logistique_courbes"}
GRAVITES = {"Gamma de référence": "gravite_gamma_reference",
            "Gamma + XGBoost": "gravite_gamma_xgboost",
            "Gamma + courbes": "gravite_gamma_courbes"}
# Scores Kaggle publics obtenus par mon binôme (fréquence, gravité) -> score
KAGGLE = {("logistique de référence", "Gamma de référence"): 1365.943,
          ("logistique de référence", "Gamma + XGBoost"): 1365.913,
          ("logistique de référence", "Gamma + courbes"): 1365.745,
          ("logistique + courbes", "Gamma + courbes"): 1365.657}


def rmse_prime(prime, cout):
    return np.sqrt(np.mean((np.asarray(prime) - np.asarray(cout)) ** 2))


lignes = [{"fréquence": "(prime constante)", "gravité": "", "RMSE locale hors pli": rmse_prime(np.full(len(oof), oof["montant_sinistre"].mean()), oof["montant_sinistre"]), "Kaggle public": np.nan}]
for nom_f, col_f in FREQUENCES.items():
    for nom_g, col_g in GRAVITES.items():
        prime = oof[col_f] * oof[col_g]
        lignes.append({"fréquence": nom_f, "gravité": nom_g,
                       "RMSE locale hors pli": rmse_prime(prime, oof["montant_sinistre"]),
                       "Kaggle public": KAGGLE.get((nom_f, nom_g), np.nan)})
combinaisons = pd.DataFrame(lignes).sort_values("RMSE locale hors pli")
combinaisons.round(4)

,fréquence,gravité,RMSE locale hors pli,Kaggle public
18,"modèle final, 14 variables (notebook 01, 4.14)",Gamma + courbes,541.7917,NaN
15,"23 variables (notebook 01, 4.13)",Gamma + courbes,541.8086,NaN
12,"actuel + métier, m = 1000 (notebook 01)",Gamma + courbes,541.8177,NaN
17,"modèle final, 14 variables (notebook 01, 4.14)",Gamma + XGBoost,541.8312,NaN
14,"23 variables (notebook 01, 4.13)",Gamma + XGBoost,541.8363,NaN
16,"modèle final, 14 variables (notebook 01, 4.14)",Gamma de référence,541.8435,NaN
11,"actuel + métier, m = 1000 (notebook 01)",Gamma + XGBoost,541.8442,NaN
13,"23 variables (notebook 01, 4.13)",Gamma de référence,541.8482,NaN
10,"actuel + métier, m = 1000 (notebook 01)",Gamma de référence,541.8575,NaN
21,logistique + courbes,Gamma + courbes,541.9266,1365.657


→ **L'ordre des RMSE locales est le même que celui des scores Kaggle** pour les 4 combinaisons notées : 542,0441 → 1365.943 ; 542,0282 → 1365.913 ; 541,9951 → 1365.745 ; 541,9266 → 1365.657. Notre évaluation locale est donc un bon guide pour choisir un modèle sans avoir à soumettre.
- **Les écarts sont minuscules** : toutes les combinaisons sont entre 541,93 et 542,04, contre 545,40 pour une prime constante.
- **La correction de la fréquence pèse même plus que celle de la gravité** : avec chacune des trois gravités, passer de la logistique de référence à la logistique + courbes fait gagner de 0,068 à 0,078 en RMSE ; passer de la Gamma de référence à Gamma + courbes ne fait gagner que 0,039 à 0,049. Améliorer la fréquence est donc une piste utile.
- **Ma logistique actuelle (notebook 01) fait un peu moins bien que la logistique de référence** sur la prime : 542,0162 contre 541,9951 avec Gamma + courbes, soit +0,021 (et +0,022 avec les deux autres gravités). Leur log-loss est pourtant presque identique (0,21402 contre 0,21399, notebook 01, 4.7). L'écart reste à tester pli par pli avant de conclure.
- **Nouveaux candidats (notebook 01, 4.12)** : « actuel + métier, $m$ = 1000 » est la meilleure fréquence avec **chacune** des trois gravités. Avec Gamma + courbes, il fait 541,8177, contre 541,9266 pour la meilleure combinaison connue (−0,109). La sélection à 6 variables fait 541,9826 : mieux que la logistique de référence, mais moins bien que la logistique + courbes.
- **Modèle final, 14 variables (notebook 01, 4.14)** : c'est la meilleure fréquence avec chacune des trois gravités. Avec Gamma + courbes, elle fait **541,7917**, la meilleure case du tableau (541,8086 pour la version à 23 variables de 4.13).

### Le meilleur nouveau candidat bat-il vraiment la meilleure combinaison connue ?

On compare, **pli par pli**, la RMSE de la prime des candidats du notebook 01 et de « logistique + courbes », tous deux avec la gravité Gamma + courbes. Même test que dans le notebook 01 : l'écart est significatif si sa moyenne dépasse deux erreurs-types.

Le **pli 1** est le jeu de validation du notebook 01 : il n'a servi à **aucun** choix de variables ni de réglage. C'est la vérification la plus honnête.

In [3]:
def rmse_par_pli(col_freq, col_grav):
    return oof.groupby("pli").apply(lambda d: rmse_prime(d[col_freq] * d[col_grav], d["montant_sinistre"]))


par_pli = pd.DataFrame({
    "logistique + courbes (1365.657)": rmse_par_pli("proba_logistique_courbes", "gravite_gamma_courbes"),
    "actuel + métier, m = 1000": rmse_par_pli("proba_metier", "gravite_gamma_courbes"),
    "sélection 6 variables": rmse_par_pli("proba_selection6", "gravite_gamma_courbes"),
    "23 variables (4.13)": rmse_par_pli("proba_metier_novice2", "gravite_gamma_courbes"),
    "14 variables (4.14)": rmse_par_pli("proba_14variables", "gravite_gamma_courbes"),
    "modèle final E, 17 variables (4.15)": rmse_par_pli("proba_final", "gravite_gamma_courbes"),
})
for nom in ["actuel + métier, m = 1000", "sélection 6 variables", "23 variables (4.13)", "14 variables (4.14)", "modèle final E, 17 variables (4.15)"]:
    par_pli[f"écart {nom}"] = par_pli[nom] - par_pli["logistique + courbes (1365.657)"]
display(par_pli.round(4))

for nom in ["actuel + métier, m = 1000", "sélection 6 variables", "23 variables (4.13)", "14 variables (4.14)", "modèle final E, 17 variables (4.15)"]:
    ecarts = par_pli[f"écart {nom}"]
    seuil = 2 * ecarts.std() / np.sqrt(len(ecarts))
    verdict = "égal" if abs(ecarts.mean()) <= seuil else ("mieux" if ecarts.mean() < 0 else "pire")
    print(f"{nom} : écart moyen {ecarts.mean():+.4f} | seuil {seuil:.4f} | plis gagnés {(ecarts < 0).sum()}/5 | verdict : {verdict}")

,logistique + courbes (1365.657),"actuel + métier, m = 1000",sélection 6 variables,23 variables (4.13),"modèle final, 14 variables (4.14)","écart actuel + métier, m = 1000",écart sélection 6 variables,écart 23 variables (4.13),"écart modèle final, 14 variables (4.14)"
pli,,,,,,,,,
1,553.5046,553.4236,553.5425,553.3762,553.2599,-0.0810,0.0379,-0.1284,-0.2447
2,559.2941,559.1337,559.1024,559.1006,559.1419,-0.1604,-0.1917,-0.1935,-0.1522
3,530.8487,530.8176,530.8384,530.6932,530.6690,-0.0312,-0.0103,-0.1555,-0.1797
4,501.6536,501.6419,502.1671,501.6683,501.6489,-0.0117,0.5135,0.0147,-0.0048
5,561.9485,561.7029,561.9231,561.8324,561.8645,-0.2455,-0.0253,-0.1160,-0.0839


actuel + métier, m = 1000 : écart moyen -0.1060 | seuil 0.0866 | plis gagnés 5/5 | verdict : mieux
sélection 6 variables : écart moyen +0.0648 | seuil 0.2373 | plis gagnés 3/5 | verdict : égal
23 variables (4.13) : écart moyen -0.1158 | seuil 0.0704 | plis gagnés 4/5 | verdict : mieux
modèle final, 14 variables (4.14) : écart moyen -0.1331 | seuil 0.0823 | plis gagnés 5/5 | verdict : mieux


→ **Le modèle final de fréquence (14 variables, notebook 01, 4.14) bat la meilleure combinaison connue dans les 5 paquets** : écart moyen −0,133, au-dessus du seuil (0,082), verdict « mieux ». Sur le paquet 1, qui n'a servi à aucun choix, il gagne −0,245.

| Fréquence (× Gamma + courbes) | Écart moyen vs « logistique + courbes » | Paquets gagnés | Verdict |
|---|---|---|---|
| actuel + métier, $m$ = 1000 (4.12) | −0,106 | 5 / 5 | mieux |
| 23 variables (4.13) | −0,116 | 4 / 5 | mieux |
| **modèle final, 14 variables (4.14)** | **−0,133** | **5 / 5** | **mieux** |
| sélection 6 variables (4.10) | +0,065 | 3 / 5 | égal |

**Prudence** :
- les variables, le lissage et la taille du modèle final ont été choisis avec les données du train ; le paquet 1 a été regardé pour choisir la taille (14 variables). Le résultat est donc un peu optimiste, et **Kaggle reste le vrai juge** ;
- pour comparaison, en local, la correction par courbes du binôme gagnait 0,0685 (541,9951 → 541,9266), et cela avait donné −0,088 sur Kaggle (1365.745 → 1365.657).

→ Le modèle final est la fréquence à soumettre : `submission_candidat.csv` (section 3).

## 2. Cohérence globale : l'assureur serait-il rentable ?

Pour la meilleure combinaison, on compare sur le train (hors pli) la **somme des primes prédites** à la **somme des sinistres observés**, au total et pli par pli. Un ratio proche de 1 signifie que les primes couvrent en moyenne le coût des sinistres.

In [4]:
FREQ_RETENUE, GRAV_RETENUE = "proba_logistique_courbes", "gravite_gamma_courbes"

oof["prime"] = oof[FREQ_RETENUE] * oof[GRAV_RETENUE]
coherence = oof.groupby("pli").agg(primes_predites=("prime", "sum"), sinistres_observes=("montant_sinistre", "sum"))
coherence.loc["total"] = coherence.sum()
coherence["ratio primes / sinistres"] = coherence["primes_predites"] / coherence["sinistres_observes"]
print(f"Prime moyenne prédite : {oof['prime'].mean():.2f} € | coût moyen observé : {oof['montant_sinistre'].mean():.2f} €")
coherence.round(3)

Prime moyenne prédite : 103.35 € | coût moyen observé : 103.37 €


,primes_predites,sinistres_observes,ratio primes / sinistres
pli,,,
1,1029215.154,1039292.53,0.990
2,1023526.867,1054345.65,0.971
3,1031746.725,1003627.18,1.028
4,1046515.213,998930.30,1.048
5,1036402.479,1072271.54,0.967
total,5167406.438,5168467.20,1.000


→ Sur l'ensemble du train, **les primes prédites couvrent les sinistres observés** : ratio de 1,000, prime moyenne de 103,35 € pour un coût moyen de 103,37 €. L'assureur serait rentable en moyenne, avant frais et marge.

Pli par pli, le ratio varie de 0,967 à 1,048 : sur 10 000 contrats, quelques gros sinistres suffisent à faire varier le coût total d'environ ± 5 %.

## 3. Soumission Kaggle

Les modèles retenus ont été **réentraînés sur les 50 000 contrats** du train avant de prédire le test. On écrit deux fichiers à la racine du projet, au format attendu (`index`, `pred`) :
- `submission.csv` : la **meilleure combinaison confirmée sur Kaggle** (logistique + courbes × Gamma + courbes, 1365.657) ;
- `submission_candidat.csv` : le **modèle final de fréquence E, 17 variables** (notebook 01, 4.15) × Gamma + courbes, meilleur en local, **pas encore soumis**.

In [5]:
def ecrire_soumission(col_freq, col_grav, fichier):
    soumission = pd.DataFrame({"index": binome_test["index"], "pred": binome_test[col_freq] * binome_test[col_grav]})
    # Contrôles du format attendu par Kaggle
    assert len(soumission) == 50000 and soumission["index"].is_unique
    assert soumission["index"].min() == 50000 and soumission["index"].max() == 99999
    assert np.isfinite(soumission["pred"]).all() and (soumission["pred"] >= 0).all()
    soumission.to_csv(RACINE / fichier, index=False)
    print(f"{fichier} écrit : {len(soumission)} lignes, prime moyenne {soumission['pred'].mean():.2f} €")
    return soumission


submission = ecrire_soumission(FREQ_RETENUE, GRAV_RETENUE, "submission.csv")
candidat = ecrire_soumission("proba_final", "gravite_gamma_courbes", "submission_candidat.csv")
print(f"Écart moyen entre les deux fichiers : {np.mean(np.abs(candidat['pred'] - submission['pred'])):.2f} € par contrat")

submission.csv écrit : 50000 lignes, prime moyenne 102.71 €
submission_candidat.csv écrit : 50000 lignes, prime moyenne 102.75 €
Écart moyen entre les deux fichiers : 7.99 € par contrat
